<a href="https://colab.research.google.com/github/gaur-avvv/Jumbled-Video-Fixing/blob/main/Video_Reconstruction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Jumbled Video Reconstruction

* upload jumbled video
* extract all frames from video
* use AI model to find similar frames
* reconstruct the correct order
* create fixed video

---
## Step 1: Install Libraries

* install pytorch
* install opencv
* install scikit-learn
* install pillow

In [1]:
#install libraries
!pip install torch torchvision --quiet
!pip install opencv-python --quiet
!pip install scikit-learn --quiet
!pip install pillow --quiet

print("all libraries installed")

all libraries installed


---
## Step 2: Upload Video

* upload your jumbled video
* file name should be jumbled_video.mp4

In [2]:
from google.colab import files

#upload video
uploaded = files.upload()

#check if uploaded
if 'jumbled_video.mp4' in uploaded:
    print("video uploaded")
else:
    print("please rename video to jumbled_video.mp4")

Saving jumbled_video.mp4 to jumbled_video.mp4
video uploaded


---
## Step 3: Import Libraries

* import opencv
* import pytorch
* import numpy
* import other tools

In [3]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
import time
import os

print("libraries imported")

libraries imported


---
## Step 4: Extract Frames

* create folder for frames
* open video file
* read frames one by one
* save each frame as image

In [4]:
import cv2
import os

#create folder
os.makedirs('/content/frames', exist_ok=True)

#open the video
video = cv2.VideoCapture("/content/jumbled_video.mp4")

#get video info
fps = int(video.get(cv2.CAP_PROP_FPS))
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"extracting {total_frames} frames...")

count = 0

#keep reading frames until we reach the end
while True:
    #read a frame
    success, frame = video.read()

    #if not read then stop
    if not success:
        break

    #save the frame as image
    cv2.imwrite(f"/content/frames/frame{count:04d}.jpg", frame)
    count += 1

    if count % 50 == 0:
        print(f"extracted {count}/{total_frames} frames...")

#close the video
video.release()

print(f"done! extracted {count} frames")

extracting 300 frames...
extracted 50/300 frames...
extracted 100/300 frames...
extracted 150/300 frames...
extracted 200/300 frames...
extracted 250/300 frames...
extracted 300/300 frames...
done! extracted 300 frames


---
## Step 5: Get Frame Files

* list all frame files
* sort by name
* store paths

In [5]:
#get all frame files
frame_folder = "/content/frames"
frame_files = sorted([f for f in os.listdir(frame_folder) if f.endswith('.jpg')])
frame_paths = [os.path.join(frame_folder, f) for f in frame_files]

print(f"found {len(frame_paths)} frames")

found 300 frames


---
## Step 6: Load AI Model

* load resnet50 model
* remove last layer
* use for getting features

In [6]:
print("loading model...")

#check gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#load resnet50
resnet50 = models.resnet50(pretrained=True)

#remove last layer
feature_extractor = nn.Sequential(*list(resnet50.children())[:-1])
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

#freeze parameters
for param in feature_extractor.parameters():
    param.requires_grad = False

print("model loaded")

loading model...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 143MB/s]


model loaded


In [7]:
print("Loading pre-trained ResNet50 model...")
print("   (This might take 30-60 seconds to download the model)")

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"   Using device: {device}")

# Load pre-trained ResNet50
# This model was trained on 1.2 million images (ImageNet dataset)
resnet50 = models.resnet50(pretrained=True)

# Remove the final classification layer
# We only want the feature extraction part
feature_extractor = nn.Sequential(*list(resnet50.children())[:-1])

# Move model to device (CPU or GPU)
feature_extractor = feature_extractor.to(device)

# Set to evaluation mode (not training)
feature_extractor.eval()

# Freeze all parameters (we won't train, just use existing weights)
for param in feature_extractor.parameters():
    param.requires_grad = False

print("\nResNet50 model loaded successfully!")
print(f"   Model has {sum(p.numel() for p in feature_extractor.parameters()):,} parameters")

Loading pre-trained ResNet50 model...
   (This might take 30-60 seconds to download the model)
   Using device: cpu

ResNet50 model loaded successfully!
   Model has 23,508,032 parameters


---
## Step 7: Setup Image Processing

* resize images
* crop to 224x224
* normalize values

In [8]:
#image preprocessing
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("preprocessing ready")

preprocessing ready


In [9]:
# Define preprocessing pipeline
# These transformations match how ResNet50 was trained

transform = transforms.Compose([
    # Step 1: Resize shortest side to 256 pixels
    transforms.Resize(256),

    # Step 2: Crop center 224×224 square
    transforms.CenterCrop(224),

    # Step 3: Convert to PyTorch tensor (also scales to [0, 1])
    transforms.ToTensor(),

    # Step 4: Normalize with ImageNet statistics
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # RGB means from ImageNet
        std=[0.229, 0.224, 0.225]    # RGB stds from ImageNet
    )
])

print("Preprocessing pipeline defined")
print("   Input: Any size RGB image")
print("   Output: 224x224x3 normalized tensor")

Preprocessing pipeline defined
   Input: Any size RGB image
   Output: 224x224x3 normalized tensor


---
## Step 8: Extract Features

* process each frame with model
* get feature vector for each
* store all features

In [10]:
print(f"extracting features from {len(frame_paths)} frames...")

all_features = []
batch_size = 32
n_batches = (len(frame_paths) + batch_size - 1) // batch_size

#process each batch
for batch_idx in range(n_batches):
    start_idx = batch_idx * batch_size
    end_idx = min(start_idx + batch_size, len(frame_paths))
    batch_paths = frame_paths[start_idx:end_idx]

    #load images
    batch_tensors = []
    for path in batch_paths:
        image = Image.open(path).convert('RGB')
        tensor = transform(image)
        batch_tensors.append(tensor)

    #process batch
    batch = torch.stack(batch_tensors).to(device)

    with torch.no_grad():
        features = feature_extractor(batch)

    features = features.squeeze().cpu().numpy()

    if len(batch_paths) == 1:
        features = features.reshape(1, -1)

    all_features.append(features)

    processed = end_idx
    print(f"processed {processed}/{len(frame_paths)} frames...", end='\r')

#combine all
features_array = np.vstack(all_features)

print(f"\ndone! got {features_array.shape[0]} feature vectors")

extracting features from 300 frames...
processed 300/300 frames...
done! got 300 feature vectors


---
## Step 9: Calculate Similarity

* compare all frames
* find which frames are similar
* create similarity matrix

In [11]:
print("computing similarity...")

#calculate similarity between all frames
similarity_matrix = cosine_similarity(features_array)

print(f"done! matrix shape: {similarity_matrix.shape}")

computing similarity...
done! matrix shape: (300, 300)


---
## Step 10: Find Start Frame

* find frames with few neighbors
* these are likely start or end frames
* pick best candidates

In [12]:
print("finding start frame...")

threshold = 0.95

#count neighbors for each frame
neighbor_counts = []
for i in range(len(similarity_matrix)):
    count = np.sum(similarity_matrix[i] > threshold) - 1
    neighbor_counts.append(count)

#get frames with fewest neighbors
start_candidates = np.argsort(neighbor_counts)[:10]

print(f"found {len(start_candidates)} candidates")

finding start frame...
found 10 candidates


---
## Step 11: Build Sequence

* start from candidate frame
* pick most similar next frame
* repeat until all frames used

In [13]:
print("reconstructing sequence...")

#function to build sequence
def build_sequence(similarity_matrix, start_idx):
    n_frames = len(similarity_matrix)
    sequence = [start_idx]
    used = {start_idx}
    current = start_idx

    for step in range(n_frames - 1):
        similarities = similarity_matrix[current].copy()

        #mark used frames
        for used_idx in used:
            similarities[used_idx] = -1

        #pick most similar frame
        next_frame = np.argmax(similarities)
        sequence.append(next_frame)
        used.add(next_frame)
        current = next_frame

    return sequence

#function to calculate score
def calculate_score(similarity_matrix, sequence):
    scores = []
    for i in range(len(sequence) - 1):
        sim = similarity_matrix[sequence[i], sequence[i+1]]
        scores.append(sim)
    return np.mean(scores)

#try each candidate
best_sequence = None
best_score = -1

for candidate in start_candidates:
    seq = build_sequence(similarity_matrix, candidate)
    score = calculate_score(similarity_matrix, seq)

    if score > best_score:
        best_score = score
        best_sequence = seq

print(f"best sequence found with score: {best_score:.4f}")

reconstructing sequence...
best sequence found with score: 0.9940


---
## Step 12: Check Direction

* check if sequence is backward
* reverse if needed

In [14]:
print("checking direction...")

#check first 30 frames
first_frame = best_sequence[0]
similarities = []

for i in range(1, min(30, len(best_sequence))):
    sim = similarity_matrix[first_frame, best_sequence[i]]
    similarities.append(sim)

#calculate trend
trend = np.polyfit(range(len(similarities)), similarities, 1)[0]

#if trend is positive, sequence is backward
if trend > 0:
    print("reversing sequence...")
    best_sequence = best_sequence[::-1]

final_score = calculate_score(similarity_matrix, best_sequence)
print(f"final score: {final_score:.4f}")

checking direction...
final score: 0.9940


---
## Step 13: Check Quality

* calculate similarity scores
* show statistics

In [15]:
#calculate consecutive similarities
consecutive_sims = []
for i in range(len(best_sequence) - 1):
    sim = similarity_matrix[best_sequence[i], best_sequence[i+1]]
    consecutive_sims.append(sim)

consecutive_sims = np.array(consecutive_sims)

print(f"mean similarity: {np.mean(consecutive_sims):.4f}")
print(f"min similarity: {np.min(consecutive_sims):.4f}")
print(f"max similarity: {np.max(consecutive_sims):.4f}")

mean similarity: 0.9940
min similarity: 0.8818
max similarity: 0.9986


---
## Step 14: Create Video

* write frames in correct order
* save as reconstructed_video.mp4

In [16]:
print("creating video...")

#get dimensions
first_frame = cv2.imread(frame_paths[0])
height, width = first_frame.shape[:2]

#create video writer
output_path = "/content/reconstructed_video.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

#write frames in correct order
for i, frame_idx in enumerate(best_sequence):
    frame = cv2.imread(frame_paths[frame_idx])
    video_writer.write(frame)

    if (i + 1) % 50 == 0:
        print(f"encoded {i + 1}/{len(best_sequence)} frames...", end='\r')

video_writer.release()

file_size = os.path.getsize(output_path) / (1024 * 1024)
print(f"\nvideo created: {output_path}")
print(f"file size: {file_size:.2f} MB")

creating video...
encoded 300/300 frames...
video created: /content/reconstructed_video.mp4
file size: 62.00 MB


---
## Step 15: Download Video

* download fixed video

In [17]:
from google.colab import files

print("downloading video...")
files.download('/content/reconstructed_video.mp4')
print("download started")

downloading video...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

download started


---
## Summary

* video reconstruction complete

In [18]:
print("done!")
print(f"extracted {len(frame_paths)} frames")
print(f"quality: {final_score*100:.2f}%")
print("output: /content/reconstructed_video.mp4")

done!
extracted 300 frames
quality: 99.40%
output: /content/reconstructed_video.mp4


---
## How It Works

* extract frames from video
* use resnet50 to analyze frames
* calculate similarity between frames
* build sequence by picking similar frames
* create video with correct order